In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:import pandas as pd

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(labels='Order_ID',axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
    df.dropna(inplace=True)
    print("\nMissing Values handled.")
    print("\nMissing Values per Column:")
    print(missing_values[missing_values > 0])

  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:

# from sklearn.preprocessing import OneHotEncoder

# categorical_cols = df.select_dtypes(include=["object"]).columns
# for col in categorical_cols:
#     print(f"Encoding column: {col}")
#     ohe = OneHotEncoder(sparse_output=False)
#     df[col] = ohe.fit_transform(df[[col]])
# df


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Encode features and target using LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # We DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': []}


n_splits = 5  # K=5 Folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)


    # Store results
    all_results[model_name]["mse"].append(mse)
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}\n\n")


In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mse = sklearn_mse(y, baseline_pred)


print(f"Baseline MSE (using mean target): {baseline_mse:.4f}")


In [ ]:
# Task 1: Write your code here:

randf_model = models["Random Forest Regressor"]

# get feature importances and list in panda series
feature_importances = randf_model.feature_importances_
importance_df = pd.Series(feature_importances, index=X.columns)

# Sort in descending
importance_df = importance_df.sort_values(ascending=False)

# Create a bar plot
plt.figure(figsize=(10, 6))
importance_df.plot(kind='bar')
plt.title('feature importances')
plt.xlabel('features')
plt.ylabel('importance')
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 6))
sns.histplot(y_pred, bins=30, kde=True, color='skyblue')
plt.title('predicted delivery time histogram')
plt.xlabel('predicted Delivery Time (minutes)')
plt.ylabel('frequency')
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:

In [ ]:
pip install catboost


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

# Initialize models for ensemble
model_rf = RandomForestRegressor(n_estimators=200, random_state=42)
model_catboost = CatBoostRegressor(random_state=42, verbose=0)

# Storage for ensemble results
ensemble_mae_scores = []

n_splits = 5  # K=5 Folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print("Starting ensemble training ...!")

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train RF
  print("Training RandomForestRegressor...")
  model_rf.fit(X_train, y_train)
  y_pred_rf = model_rf.predict(X_test)

  # Train CatBoost
  print("Training CatBoostRegressor...")
  model_catboost.fit(X_train, y_train)
  y_pred_catboost = model_catboost.predict(X_test)

  # Average predictions
  y_pred_ensemble = (y_pred_rf + y_pred_catboost) / 2

  # Calculateing MAE for ensemble
  mae_ensemble = mean_absolute_error(y_test, y_pred_ensemble)
  ensemble_mae_scores.append(mae_ensemble)
  print(f"Ensemble MAE for Fold {fold_idx + 1}: {mae_ensemble:.4f}")

# Print the averaged MAE across all folds for the ensemble
print(f"\nAverage Ensemble MAE across all folds: {np.mean(ensemble_mae_scores):.4f}")
